# שיעור 01 - מבוא לסוכני בינה מלאכותית

ברוכים הבאים לשיעור הראשון בקורס **סוכני בינה מלאכותית למתחילים**!

**סוכן בינה מלאכותית** הוא תוכנית שמשתמשת במודל שפה גדול (LLM) כמנוע ההסקה שלה ויכולה לנקוט *פעולות* בעולם האמיתי — קריאה ל-APIs, שאילתות במסדי נתונים, או הרצת קוד — כדי להשיג מטרה בשם המשתמש.

במחברת זו תבנו את הסוכן הראשון שלכם: **סוכן נסיעות** שממליץ על יעדי חופשה. לאורך הדרך תלמדו כיצד:

1. להתחבר לשירות Microsoft Foundry Agent באמצעות **מסגרת הסוכנים של Microsoft**.
2. לתת לסוכן **כלי** — פונקציית Python פשוטה שאותה הוא יכול לקרוא.
3. להריץ את הסוכן ולבחון את תגובתו.
4. להזרים את תגובת הסוכן טוקן-אחר-טוקן.


## הגדרה

לפני הפעלת פנקס זה, ודא שיש לך:

1. **פרויקט Microsoft Foundry** עם מודל צ'אט פרוס (למשל `gpt-4.1-mini`).
2. **התחברת עם Azure CLI** — הפעל `az login` במסוף שלך.
3. **הגדר את משתני הסביבה הנדרשים:**
   - `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Microsoft Foundry שלך.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהורץ.

התא למטה מתקין את חבילות הפייתון שאתה צריך.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## יצירת הסוכן הראשון שלך

לסוכן יש צורך בשני דברים:

- **הוראות** שמספרות לו *מי הוא* ו*איך להתנהג* (פירומט מערכת).
- **כלים** — פונקציות פייתון המעוטרות ב- `@tool` שהסוכן יכול לקרוא כדי לקבל מידע או לבצע פעולות.

למטה אנו מגדירים כלי פשוט שמחזיר רשימה של יעדי חופשה פופולריים. הסוכן ישתמש בכלי זה כשהמשתמש יבקש המלצות טיול.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## תגובות סטרימינג

לחוויה אינטראקטיבית יותר, ניתן **לשדר בשידור חי** את תגובת הסוכן. במקום להמתין לתשובה המלאה, הסוכן מחזיר קטעי טקסט ברגע שהם נוצרים. זה שימושי במיוחד בממשקי צ'אט שבהם רוצים להציג פלט בזמן אמת.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## סיכום

בשיעור זה למדת כיצד:

- **ליצור ספק** שמתחבר לשירות Microsoft Foundry Agent דרך `FoundryChatClient`.
- **להגדיר כלי** תוך שימוש ב-`@tool` כך שהסוכן יוכל לקרוא לפונקציות הפייתון שלך.
- **להפעיל את הסוכן** עם הודעת משתמש ולהדפיס את תגובתו.
- **לספק תגובות בזרם** לתוצר בזמן אמת.

בשיעור הבא נחקור את המסגרות האייג׳נטיות ביתר עומק ונלמד כיצד לתת לסוכנים כלים רבי עוצמה ויכולות הסקת מסקנות רב-שלביות.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
